## Identify all classic keto-enol H transfers from wb97xd3 to perform conformer searches

Change `RMG-database/input/kinetics/families/ketoenol/groups.py` line 32 to be
```
entry(
    index = 0,
    label = "Root",
    group = 
"""
1 *2 R!H   u0 {2,S} {3,D}
2 *3 [O,S] u0 {1,S} {4,S}
3 *1 R!H   u0 {1,D}
4 *4 H     u0 {2,S}
""",
    kinetics = None,
)
```

In [1]:
import os
import time

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdMolDescriptors import CalcNumRotatableBonds
from rdkit.Chem.Lipinski import RotatableBondSmarts

from rmgpy import settings
from rmgpy.data.rmg import RMGDatabase
from rmgpy.exceptions import AtomTypeError
import rmgpy.molecule.element as elements
from rmgpy.molecule.molecule import Atom, Bond, Molecule

from arc.parser import parse_xyz_from_file
from arc.plotter import draw_3d, draw_structure, plot_3d_mol_as_scatter, show_sticks
from arc.species.converter import get_xyz_radius, xyz_from_data, xyz_to_str, xyz_to_x_y_z

from rdmc.mol import RDKitMol
from rdmc.ts import get_all_changing_bonds, get_formed_and_broken_bonds
from rdmc.view import mol_viewer

In [2]:
settings

{'database.directory': '/Users/kevin/Downloads/RMG/RMG-database/input',
 'test_data.directory': '/Users/kevin/Downloads/RMG/RMG-Py/rmgpy/test_data'}

In [3]:
FINAL_CSV_FILES = '/Users/kevin/Dropbox (MIT)/kinetics_database_paper/FINAL_CSV_FILES_divide_1014'
wb97xd3_log_files =  '/Users/kevin/Dropbox (MIT)/kinetics_database_paper/wb97xd3/qm_logs'
ccsdtf12_log_files = '/Users/kevin/Dropbox (MIT)/kinetics_database_paper/ccsdtf12/qm_logs'

In [4]:
path = os.path.join(FINAL_CSV_FILES, 'ccsdtf12_dz', 'ccsdtf12_dz.csv')
df_wb97xd3 = pd.read_csv(path)
df_wb97xd3

,idx,rsmi,psmi,dE0,dHrxn298,rmg_family
0,0,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([C:2]([N:3]=[O:4])=[N+:6]=[N-:5])([H:7])...,48.61085,26.80354,NaN
1,1,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([N:3]=[C:2]=[N:6][N:5]=[O:4])([H:7])([H:...,74.02980,28.82627,NaN
2,2,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]1([H:6])([H:7])[O:2][C:3]([H:9])([H:10])[...,97.42200,12.61595,NaN
3,3,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]1([...,75.25375,29.00269,NaN
4,4,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]([C...,72.16356,1.47560,NaN
...,...,...,...,...,...,...
11921,11956,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C:2][C:4]([O:5][C:6](=[O:7])[H:15])([H:...,75.56813,79.68294,NaN
11922,11957,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]1([H:11])[O:3][C@:6]([O:7][H:12])...,42.41621,5.75585,NaN
11923,11958,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]([O:3][H:12])([C:4](=[O:5])[H:14]...,72.75039,30.60531,NaN
11924,11959,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1](=[C:2]([C:4]([O:5][C:6](=[O:7])[H:15])([...,65.83112,14.53918,"1,3_Insertion_ROR"


In [5]:
df_wb97xd3.rmg_family.isna().sum()

10445

In [6]:
df_wb97xd3.rmg_family.value_counts()

ketoenol                                    687
Singlet_Carbene_Intra_Disproportionation    256
1,3_Insertion_ROR                           195
Retroene                                    173
2+2_cycloaddition                            61
1,2_Insertion_CO                             39
Intra_2+2_cycloaddition_Cd                   18
Intra_ene_reaction                           14
1,3_NH3_elimination                          12
1,2_Insertion_carbene                         8
1+2_Cycloaddition                             7
6_membered_central_C-C_shift                  4
1,3_Insertion_CO2                             4
Diels_alder_addition                          3
Name: rmg_family, dtype: int64

In [7]:
df_wb97xd3.rmg_family.value_counts().sum()

1481

In [8]:
df = df_wb97xd3.drop(['rmg_family'], axis=1)
df

,idx,rsmi,psmi,dE0,dHrxn298
0,0,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([C:2]([N:3]=[O:4])=[N+:6]=[N-:5])([H:7])...,48.61085,26.80354
1,1,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([N:3]=[C:2]=[N:6][N:5]=[O:4])([H:7])([H:...,74.02980,28.82627
2,2,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]1([H:6])([H:7])[O:2][C:3]([H:9])([H:10])[...,97.42200,12.61595
3,3,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]1([...,75.25375,29.00269
4,4,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]([C...,72.16356,1.47560
...,...,...,...,...,...
11921,11956,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C:2][C:4]([O:5][C:6](=[O:7])[H:15])([H:...,75.56813,79.68294
11922,11957,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]1([H:11])[O:3][C@:6]([O:7][H:12])...,42.41621,5.75585
11923,11958,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]([O:3][H:12])([C:4](=[O:5])[H:14]...,72.75039,30.60531
11924,11959,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1](=[C:2]([C:4]([O:5][C:6](=[O:7])[H:15])([...,65.83112,14.53918


Helper functions

In [9]:
def load_rmg_database(kinetics_families='default'):
    """
    Helper function to load the RMG-database.

    Returns:
        An instance of RMG-database
    """
    database_path = settings['database.directory']
    database = RMGDatabase()
    database.load(path=database_path,
                  thermo_libraries=['primaryThermoLibrary'],
                  reaction_libraries=[],
                  seed_mechanisms=[],
                  kinetics_families=kinetics_families,
                  )
    return database

In [10]:
def from_rdkit_mol(rdkit_mol, sort=False, raise_atomtype_exception=True):
    """
    Converts an RDKit Molecule object to an RMG-Py Molecule object. 

    Args:
        rdkit_mol: An RDKit Molecule object
        sort: boolean indicating whether to sort the atoms in the new RMG-Py Molecule object.
              atoms are sorted by placing heaviest atoms first and H atoms last
        raise_atomtype_exception
    """
    bond_order_dict = {'SINGLE': 1, 'DOUBLE': 2, 'TRIPLE': 3, 'QUADRUPLE': 4, 'AROMATIC': 1.5}

    mol = Molecule()
    mol.vertices = []

    rdkit_mol.UpdatePropertyCache(strict=False)
    Chem.rdmolops.Kekulize(rdkit_mol, clearAromaticFlags=True)

    # iterate through atoms
    for i in range(rdkit_mol.GetNumAtoms()):
        rdkit_atom = rdkit_mol.GetAtomWithIdx(i)

        # use atomic number as key for element
        number = rdkit_atom.GetAtomicNum()
        isotope = rdkit_atom.GetIsotope()
        element = elements.get_element(number, isotope or -1)

        # process charge
        charge = rdkit_atom.GetFormalCharge()
        radical_electrons = rdkit_atom.GetNumRadicalElectrons()
        atom = Atom(element, radical_electrons, charge, '', 0)
        mol.vertices.append(atom)

        # add bonds by iterating again through atoms
        for j in range(0, i):
            rdkit_bond = rdkit_mol.GetBondBetweenAtoms(i, j)
            if rdkit_bond is not None:
                # process bond type
                rd_bond_type = rdkit_bond.GetBondType()
                order = bond_order_dict[rd_bond_type.name]
                bond = Bond(mol.vertices[i], mol.vertices[j], order)
                mol.add_bond(bond)

    # update lone pairs first because the charge was set by RDKit
    mol.update_lone_pairs()

    # update the atom type, charge, and multiplicity
    mol.update(raise_atomtype_exception=raise_atomtype_exception, sort_atoms=sort)

    return mol

In [11]:
def find_reaction_family(database, reactants, products, verbose=True):
    """
    Helper function for finding RMG reaction families when given a set of reactants and products.

    Args:
        database: an instance of RMG database.
        reactants: list of reactant molecules as RMG Molecule objects
        products: list of product molecules as RMG Molecule objects
        verbose: boolean indicating whether to print results

    Returns:
        (family_label, is_forward). None if no match.
    """
    
    # see if RMG can find this reaction
    for family in database.kinetics.families.values():
        family.save_order = False
    reaction_list = database.kinetics.generate_reactions(reactants=[mol.copy() for mol in reactants],
                                                         products=[mol.copy() for mol in products])
    # get reaction information
    for rxn in reaction_list:
        family, forward = rxn.family, rxn.is_forward
        if verbose:
            print(f'{rxn}\n',
                  f'RMG family: {family}\n',
                  f'Is forward reaction: {forward}')
        return family, forward
    else:
        if verbose:
            print("Doesn't match any RMG reaction family!")

Identify classic keto enol H transfers

In [12]:
# load RMG-database
database = load_rmg_database('Ketoenol')

In [13]:
# create new columns for storing RMG reaction family and direction
a = np.empty(df.shape[0])
a[:] = np.nan
df.insert(df.shape[1], "rmg_family", a)
df.insert(df.shape[1], "forward", a)
df

,idx,rsmi,psmi,dE0,dHrxn298,rmg_family,forward
0,0,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([C:2]([N:3]=[O:4])=[N+:6]=[N-:5])([H:7])...,48.61085,26.80354,NaN,NaN
1,1,[C:1]([c:2]1[n:3][o:4][n:5][n:6]1)([H:7])([H:8...,[C:1]([N:3]=[C:2]=[N:6][N:5]=[O:4])([H:7])([H:...,74.02980,28.82627,NaN,NaN
2,2,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]1([H:6])([H:7])[O:2][C:3]([H:9])([H:10])[...,97.42200,12.61595,NaN,NaN
3,3,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]1([...,75.25375,29.00269,NaN,NaN
4,4,[C:1]([O:2][C:3]([C:4]([O:5][H:13])([H:11])[H:...,[C:1]([O:2][H:13])([H:6])([H:7])[H:8].[C:3]([C...,72.16356,1.47560,NaN,NaN
...,...,...,...,...,...,...,...
11921,11956,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C:2][C:4]([O:5][C:6](=[O:7])[H:15])([H:...,75.56813,79.68294,NaN,NaN
11922,11957,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]1([H:11])[O:3][C@:6]([O:7][H:12])...,42.41621,5.75585,NaN,NaN
11923,11958,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1]([C@@:2]([O:3][H:12])([C:4](=[O:5])[H:14]...,72.75039,30.60531,NaN,NaN
11924,11959,[C:1]([C@@:2]([O:3][H:12])([C:4]([O:5][C:6](=[...,[C:1](=[C:2]([C:4]([O:5][C:6](=[O:7])[H:15])([...,65.83112,14.53918,NaN,NaN


The rows you get back from iterrows are copies that are no longer connected to the original data frame, so edits don't change your dataframe. Thankfully, because each item you get back from iterrows contains the current index, you can use that to access and edit the relevant row of the dataframe:

In [14]:
# find any classic ketoenol matches
start = time.time()
for i, row in df.iterrows():
    try:
        # reactant
        rsmile = row.rsmi
        r_mol = Chem.MolFromSmiles(rsmile, sanitize=False)
        reactant_mols = [from_rdkit_mol(r_mol)]

        # product/s
        psmiles = row.psmi
        p_mols = [Chem.MolFromSmiles(psmi, sanitize=False) for psmi in psmiles.split('.')]
        product_mols = [from_rdkit_mol(p_mol) for p_mol in p_mols] 

        # find the reaction in the RMG-database if it matches any existing templates
        family_label = None
        try:
            family_label, forward = find_reaction_family(database,
                                                         reactant_mols,
                                                         product_mols,
                                                         verbose=False)
            
            # print(row.idx, family_label)
        except TypeError:
            # cannot find any matches
            pass      

        if family_label:
            df.loc[i, 'rmg_family'] = family_label
            df.loc[i, 'forward'] = forward
    
    except AtomTypeError:
        print(f'rxn{row.idx:06} had an AtomTypeError')

num_RMG_reactions = len(df[~df.rmg_family.isna()])
print(f'Number of RMG reactions: {num_RMG_reactions}')
print(f'Elapsed time: {time.time() - start:.2f} seconds')

ERROR:root:Could not update atomtypes for this molecule:
multiplicity -187
1 N u0 p1 c0 {2,S} {6,S} {7,S}
2 C u0 p0 c+2 {1,S} {3,S}
3 N u0 p1 c0 {2,S} {4,S} {5,S}
4 N u0 p2 c0 {3,S}
5 H u0 p0 c0 {3,S}
6 H u0 p0 c0 {1,S}
7 H u0 p0 c0 {1,S}



rxn000362 had an AtomTypeError


ERROR:root:Could not update atomtypes for this molecule:
multiplicity -187
1  O u0 p2 c0 {2,S} {11,S}
2  C u0 p0 c0 {1,S} {3,S} {4,S} {10,S}
3  H u0 p0 c0 {2,S}
4  C u0 p0 c0 {2,S} {5,S} {6,S} {7,S}
5  H u0 p0 c0 {4,S}
6  H u0 p0 c0 {4,S}
7  C u0 p1 c0 {4,S} {8,S}
8  N u0 p1 c0 {7,S} {9,S} {10,S}
9  H u0 p0 c0 {8,S}
10 C u0 p0 c+2 {2,S} {8,S}
11 H u0 p0 c0 {1,S}



rxn000462 had an AtomTypeError


ERROR:root:Could not update atomtypes for this molecule:
multiplicity -187
1  N u0 p1 c0 {2,S} {10,S} {11,S}
2  C u0 p0 c+2 {1,S} {3,S}
3  C u0 p0 c0 {2,S} {4,S} {5,S} {9,S}
4  H u0 p0 c0 {3,S}
5  C u0 p0 c+1 {3,S} {6,S} {7,S}
6  H u0 p0 c0 {5,S}
7  N u0 p1 c0 {5,S} {8,S} {9,S}
8  H u0 p0 c0 {7,S}
9  N u0 p2 c-1 {3,S} {7,S}
10 H u0 p0 c0 {1,S}
11 H u0 p0 c0 {1,S}



...

rxn007182 had an AtomTypeError


ERROR:root:Could not update atomtypes for this molecule:
multiplicity -187
1  C u0 p0 c0 {2,S} {12,S} {13,S} {14,S}
2  C u0 p0 c0 {1,S} {3,S} {6,S} {11,S}
3  C u0 p0 c0 {2,S} {4,S} {5,S} {6,S}
4  H u0 p0 c0 {3,S}
5  H u0 p0 c0 {3,S}
6  C u0 p0 c0 {2,S} {3,S} {7,S} {8,S}
7  H u0 p0 c0 {6,S}
8  C u0 p1 c0 {6,S} {9,S}
9  N u0 p1 c0 {8,S} {10,S} {11,S}
10 H u0 p0 c0 {9,S}
11 C u0 p0 c+2 {2,S} {9,S}
12 H u0 p0 c0 {1,S}
13 H u0 p0 c0 {1,S}
14 H u0 p0 c0 {1,S}



rxn007634 had an AtomTypeError


ERROR:root:Could not update atomtypes for this molecule:
multiplicity -187
1 N u0 p2 c0 {2,S}
2 N u0 p1 c0 {1,S} {3,S} {6,S}
3 C u0 p0 c+2 {2,S} {4,S}
4 O u0 p2 c0 {3,S} {5,S}
5 H u0 p0 c0 {4,S}
6 H u0 p0 c0 {2,S}



rxn011451 had an AtomTypeError
Number of RMG reactions: 361
Elapsed time: 41.96 seconds


In [15]:
df.to_csv('ccsdtf12_classic_ketoenol_H_transfer_all.csv', index=False)

In [16]:
df[df.rmg_family == 'Ketoenol']

,idx,rsmi,psmi,dE0,dHrxn298,rmg_family,forward
16,17,[O:1]([c:2]1=[n:3][c:4]([H:8])[n:5][n:6]1[H:9]...,[O:1]=[c:2]1[n:3]([H:7])[c:4]([H:8])[n:5][n:6]...,43.97984,-5.55559,Ketoenol,True
47,49,[C:1](/[N:2]=[C:3](/[N:4]([C:5](=[O:6])[H:12])...,[C:1](/[N:2]=[C:3](/[N:4]=[C:5](/[O:6][H:11])[...,45.24489,13.29720,Ketoenol,False
75,77,[C:1]([C:2]([C:3]([C:4](=[O:5])[H:13])([H:11])...,[C:1]([C:2](/[C:3](=[C:4](\[O:5][H:11])[H:13])...,71.02608,9.27851,Ketoenol,False
139,141,[O:1]=[C:2]1[C:3]([H:7])([H:8])[C@@:4]2([H:9])...,[O:1]([C:2]1=[C:6]2[C@@:4]([H:9])([C:3]1([H:7]...,79.53759,54.55761,Ketoenol,False
142,144,[O:1]=[C:2]1[C:3]([H:7])([H:8])[C@@:4]2([H:9])...,[O:1]([C:2]1=[C:3]([H:7])[C@@:4]2([H:9])[O:5][...,79.85719,26.11012,Ketoenol,False
...,...,...,...,...,...,...,...
11537,11571,[C:1]([C:2]([C@@:3]([C:4]([H:14])([H:15])[H:16...,[C:1]([C:2]([C@@:3]([C:4]([H:14])([H:15])[H:16...,43.39747,10.37817,Ketoenol,False
11551,11585,[C:1]([C@@:2]1([H:11])[N:3]([H:12])[C:4]([H:13...,[C:1]([C@@:2]1([H:11])[N:3]([H:12])[C:4]([H:13...,74.74778,14.24783,Ketoenol,True
11597,11631,[C:1]([C@:2]12[C:3]([H:11])([H:12])[C@@:4]1([H...,[C:1]([C@:2]12[C:3]([H:11])([H:12])[C:4]1=[C:5...,87.37352,61.23769,Ketoenol,False
11738,11772,[C:1]([C:2]1([C:3]([H:11])([H:12])[H:13])[C:4]...,[C:1]([C:2]1([C:3]([H:11])([H:12])[H:13])[C:4]...,60.14076,15.40731,Ketoenol,False


In [17]:
df[df.rmg_family == 'Ketoenol'].to_csv('ccsdtf12_classic_ketoenol_H_transfer.csv', index=False)

In [19]:
# barrier height is very reasonable
df[df.rmg_family == 'Ketoenol'].dE0.describe()

count    361.000000
mean      62.751709
std       12.842705
min       28.808580
25%       51.057030
50%       65.949680
75%       71.285590
max       92.629380
Name: dE0, dtype: float64

In [20]:
# barrier height is much higher for all other reactions
df[df.rmg_family != 'Ketoenol'].dE0.describe()

count    11565.000000
mean        80.601198
std         21.852401
min          9.418760
25%         65.427350
50%         79.324240
75%         93.754860
max        195.598150
Name: dE0, dtype: float64

In [22]:
df_ketoenol = pd.read_csv('ccsdtf12_classic_ketoenol_H_transfer.csv')
# make sure all reactions are unimolecular
for i, row in df_ketoenol.iterrows():
    rsmi = row.rsmi
    num_reactants = len(rsmi.split('.'))
    assert num_reactants == 1
    
    psmi = row.psmi
    num_products = len(psmi.split('.'))
    assert num_products == 1